In [19]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import PeftModel
import re


# Prepare the dataset and language model

### Import Data and Sanity Check

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [3]:
DATA_PATH = Path("/content/drive/MyDrive/Colab Notebooks/data/SMSSpamCollection")

In [4]:
rows = []
with DATA_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.rstrip("\n")
        if not line:
            continue
        label, text = line.split("\t", 1)
        rows.append({"label": label, "text": text})

df = pd.DataFrame(rows)


In [5]:
df.head(3)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


In [6]:
print("\nShape:", df.shape)
print("\nHead:")
display(df.head())

print("\nLabel counts:")
print(df["label"].value_counts(dropna=False))

print("\nAny nulls?")
print(df.isna().sum())

print("\nAny empty texts?")
print((df["text"].str.strip() == "").sum())

print("\nText length summary:")
display(df["text"].str.len().describe())


Shape: (5574, 2)

Head:


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Label counts:
label
ham     4827
spam     747
Name: count, dtype: int64

Any nulls?
label    0
text     0
dtype: int64

Any empty texts?
0

Text length summary:


,text
count,5574.000000
mean,80.478292
std,59.848302
min,2.000000
25%,36.000000
50%,62.000000
75%,122.000000
max,910.000000


### Build Q/A format

In [7]:
# 2) Create instruction-style Q/A pairs
SYSTEM_PROMPT = "You are a helpful assistant that classifies SMS messages."
QUESTION_TEMPLATE = "Classify the following SMS as spam or ham.\n\nSMS: {text}\n\nAnswer with exactly one word: spam or ham."

df["question"] = df["text"].apply(lambda t: QUESTION_TEMPLATE.format(text=t))
df["answer"] = df["label"]


In [8]:
qa_df = df[["question", "answer"]].copy()

print("QA shape:", qa_df.shape)
print("\nAnswer counts:")
print(qa_df["answer"].value_counts())

print("\nExample Q/A:")
display(qa_df.head(3))

QA shape: (5574, 2)

Answer counts:
answer
ham     4827
spam     747
Name: count, dtype: int64

Example Q/A:


,question,answer
0,Classify the following SMS as spam or ham.\n\n...,ham
1,Classify the following SMS as spam or ham.\n\n...,ham
2,Classify the following SMS as spam or ham.\n\n...,spam


In [9]:
test_size = 15

train_df, test_df = train_test_split(
    qa_df,
    test_size=test_size,
    stratify=qa_df["answer"],
    random_state=42
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("\nTrain label counts:")
print(train_df["answer"].value_counts())

print("\nTest label counts:")
print(test_df["answer"].value_counts())

print("\nSample test examples:")
display(test_df.head(5))


Train: (5559, 2)
Test: (15, 2)

Train label counts:
answer
ham     4814
spam     745
Name: count, dtype: int64

Test label counts:
answer
ham     13
spam     2
Name: count, dtype: int64

Sample test examples:


,question,answer
3971,Classify the following SMS as spam or ham.\n\n...,ham
4364,Classify the following SMS as spam or ham.\n\n...,ham
913,Classify the following SMS as spam or ham.\n\n...,ham
1959,Classify the following SMS as spam or ham.\n\n...,ham
3366,Classify the following SMS as spam or ham.\n\n...,ham


### Choose Base Model + Load Tokenizer

In [10]:
checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32).to(device)

print("Model loaded.")
print("Device:", next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded.
Device: cuda:0


The SMSSpamCollection dataset (5,574 SMS messages) was loaded and converted into instruction-style question/answer pairs. Each question asks the model to classify an SMS as spam or ham, and the expected output is a single-word label.

A stratified subset of 15 examples was held out for evaluation (not used during fine-tuning). The remaining 5,559 examples were used for training.

For fine-tuning, I selected `HuggingFaceTB/SmolLM2-1.7B-Instruct`, a HuggingFace model within the 1.7B parameter limit required by the assignment.

# 2. Show sample outputs with the base language model

In [12]:
questions = test_df["question"].tolist()
gold = test_df["answer"].tolist()

print("Num test questions:", len(questions))
print("Gold label counts:")
print(pd.Series(gold).value_counts())


Num test questions: 15
Gold label counts:
ham     13
spam     2
Name: count, dtype: int64


In [15]:
tokenizer_small = tokenizer
model_small = model

tokenizer_small.add_eos_token = True
tokenizer_small.pad_token_id = 0
tokenizer_small.padding_side = "left"

generator_params = {
    "max_new_tokens": 250,
    "temperature": 0.1,
    "top_p": 0.9,
    "do_sample": True,
    "pad_token_id": tokenizer_small.eos_token_id,
    "eos_token_id": tokenizer_small.eos_token_id
}

def query_lm(model, tokenizer, question, generator_params):
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False)

    enc = tokenizer(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            **generator_params
        )
    return tokenizer_small.decode(outputs[0])


In [17]:
for i in range(3):
    out = query_lm(model_small, tokenizer_small, questions[i], generator_params)
    print(f"\n--- Example {i+1} ---")
    print("GOLD:", gold[i])
    print("BASE OUTPUT:\n", out)




--- Example 1 ---
GOLD: ham
BASE OUTPUT:
 <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the following SMS as spam or ham.

SMS: That's the trouble with classes that go well - you're due a dodgey one … Expecting mine tomo! See you for recovery, same time, same place 

Answer with exactly one word: spam or ham.<|im_end|>
<|im_end|>assistant
ham<|im_end|>

--- Example 2 ---
GOLD: ham
BASE OUTPUT:
 <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the following SMS as spam or ham.

SMS: Evry Emotion dsn't hav Words.Evry Wish dsn't hav Prayrs.. If u Smile,D World is wit u.Othrwise even d Drop of Tear dsn't lik 2 Stay wit u.So b happy.. Good morning, keep smiling:-)

Answer with exactly one word: spam or ham.<|im_end|>
<|im_end|>assistant
spam<|im_end|>

--- Example 3 ---
GOLD: ham
BASE OUTPUT:
 <|im_start|>system
You are a helpful AI assist

In [20]:
# quick demo
def extract_label(decoded_text):
    matches = re.findall(r"\b(spam|ham)\b", decoded_text.lower())
    return matches[-1] if matches else "UNKNOWN"

for i in range(3):
    print(i+1, "GOLD:", gold[i], "PRED:", extract_label(outputs_base[i]))


1 GOLD: ham PRED: ham
2 GOLD: ham PRED: spam
3 GOLD: ham PRED: ham


Using the 15 held-out evaluation questions, I queried the base HuggingFace model (`HuggingFaceTB/SmolLM2-1.7B-Instruct`) following the workflow (`apply_chat_template` → tokenize → `model.generate`).

The base model is sub-optimal on this SMS spam classification task. For example, it incorrectly predicts `spam` for a normal conversational message and also misclassifies some spam-like promotional messages as `ham`. This establishes a clear baseline before applying LoRA fine-tuning.

# 3. LoRA fine-tuning with PEFT

In [21]:
train_questions = train_df["question"].tolist()
train_answers = train_df["answer"].tolist()

qa_dict = {
    "question": train_questions,
    "answer": train_answers
}

train_dataset = Dataset.from_dict(qa_dict)
print(train_dataset)


Dataset({
    features: ['question', 'answer'],
    num_rows: 5559
})


In [22]:
max_length = 256

def tokenize_qa(example):
    text = example["question"] + "\n" + example["answer"]
    tokenized = tokenizer_small(
        text,
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_qa, remove_columns=["question", "answer"])
print(tokenized_train)


Map:   0%|          | 0/5559 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5559
})


In [23]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

peft_model = get_peft_model(model_small, lora_config)
peft_model.print_trainable_parameters()


trainable params: 1,572,864 || all params: 1,712,949,248 || trainable%: 0.0918


In [24]:
output_dir = "./smollm2_spam_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)

print(training_args)


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

In [25]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train
)

train_result = trainer.train()
print(train_result)


Step,Training Loss
25,17.200211
50,17.091368
75,17.064788
100,17.010625
125,17.033503
150,17.073411
175,17.000432
200,17.065457
225,16.874034
250,16.976211


TrainOutput(global_step=348, training_loss=17.03027562985475, metrics={'train_runtime': 763.1415, 'train_samples_per_second': 7.284, 'train_steps_per_second': 0.456, 'total_flos': 1.3766703524610048e+16, 'train_loss': 17.03027562985475, 'epoch': 1.0})


In [28]:
adapter_dir = "/content/drive/MyDrive/Colab Notebooks/Model/smollm2_spam_lora_adapter"

peft_model.save_pretrained(adapter_dir)
tokenizer_small.save_pretrained(adapter_dir)

print("Saved permanently to:", adapter_dir)

Saved permanently to: /content/drive/MyDrive/Colab Notebooks/Model/smollm2_spam_lora_adapter


I fine-tuned `HuggingFaceTB/SmolLM2-1.7B-Instruct` using HuggingFace’s PEFT LoRA framework, following the workflow (`LoraConfig` → `get_peft_model` → `Trainer`).

The training dataset was tokenized into `input_ids`, `attention_mask`, and `labels`, and the model was trained for 1 epoch (batch size = 4, gradient accumulation = 4, learning rate = 2e-4). Only ~0.09% of parameters were trainable, confirming parameter-efficient adaptation.

The trained LoRA adapter and tokenizer were saved permanently to Google Drive for evaluation and comparison against the base model.

# 4. Show sample outputs with fine-tuned model

In [29]:
checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
adapter_path = "/content/drive/MyDrive/Colab Notebooks/Model/smollm2_spam_lora_adapter"

# Reload base tokenizer + base model
tokenizer_ft = AutoTokenizer.from_pretrained(checkpoint)
base_model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.float32).to(device)

# Attach LoRA adapter
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()

print("Fine-tuned model loaded.")
print("Adapter path:", adapter_path)
print("Device:", next(ft_model.parameters()).device)


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Fine-tuned model loaded.
Adapter path: /content/drive/MyDrive/Colab Notebooks/Model/smollm2_spam_lora_adapter
Device: cuda:0


In [30]:
# Generate outputs from fine-tuned model on the SAME held-out questions
outputs_ft = [query_lm(ft_model, tokenizer_small, q, generator_params) for q in questions]

# Print first 3 examples
for i in range(3):
    print(f"\n--- FT Example {i+1} ---")
    print("QUESTION:\n", questions[i])
    print("GOLD:", gold[i])
    print("FT OUTPUT:\n", outputs_ft[i])
    print("FT PRED:", extract_label(outputs_ft[i]))



--- FT Example 1 ---
QUESTION:
 Classify the following SMS as spam or ham.

SMS: That's the trouble with classes that go well - you're due a dodgey one … Expecting mine tomo! See you for recovery, same time, same place 

Answer with exactly one word: spam or ham.
GOLD: ham
FT OUTPUT:
 <|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the following SMS as spam or ham.

SMS: That's the trouble with classes that go well - you're due a dodgey one … Expecting mine tomo! See you for recovery, same time, same place 

Answer with exactly one word: spam or ham.<|im_end|>
<|im_end|>assistant
ham<|im_end|>
FT PRED: ham

--- FT Example 2 ---
QUESTION:
 Classify the following SMS as spam or ham.

SMS: Evry Emotion dsn't hav Words.Evry Wish dsn't hav Prayrs.. If u Smile,D World is wit u.Othrwise even d Drop of Tear dsn't lik 2 Stay wit u.So b happy.. Good morning, keep smiling:-)

Answer with exactly one word: spam or ham.
GO

In [33]:
# Predictions (clean labels)
pred_base = [extract_label(x) for x in outputs_base]
pred_ft = [extract_label(x) for x in outputs_ft]

# Accuracy
acc_base = sum(p == y for p, y in zip(pred_base, gold)) / len(gold)
acc_ft = sum(p == y for p, y in zip(pred_ft, gold)) / len(gold)

print("Base accuracy:", acc_base)
print("FT accuracy:", acc_ft)

print("\nExamples where prediction changed (Base -> FT):")
for i, (y, pb, pf) in enumerate(zip(gold, pred_base, pred_ft), 1):
    if pb != pf:
        print(f"#{i}: GOLD={y} | BASE={pb} -> FT={pf}")


Base accuracy: 0.8
FT accuracy: 0.8666666666666667

Examples where prediction changed (Base -> FT):
#2: GOLD=ham | BASE=spam -> FT=ham


In [35]:
for i, y in enumerate(gold):
    if y == "spam":
        print(f"\n--- Spam Case #{i+1} ---")
        print("QUESTION:\n", questions[i])
        print("GOLD:", y)
        print("BASE PRED:", pred_base[i])
        print("FT PRED:", pred_ft[i])



--- Spam Case #11 ---
QUESTION:
 Classify the following SMS as spam or ham.

SMS: Sorry I missed your call let's talk when you have the time. I'm on 07090201529

Answer with exactly one word: spam or ham.
GOLD: spam
BASE PRED: ham
FT PRED: ham

--- Spam Case #12 ---
QUESTION:
 Classify the following SMS as spam or ham.

SMS: Double Mins & Double Txt & 1/2 price Linerental on Latest Orange Bluetooth mobiles. Call MobileUpd8 for the very latest offers. 08000839402 or call2optout/LF56

Answer with exactly one word: spam or ham.
GOLD: spam
BASE PRED: ham
FT PRED: ham


### Conclusion

Using the same 15 held-out test questions (not used during training), I evaluated the LoRA fine-tuned model and compared it to the base model.

- **Base accuracy:** 0.80 (12/15)
- **Fine-tuned accuracy:** 0.867 (13/15)

The fine-tuned model corrected a false positive error (Example #2: ham misclassified as spam), demonstrating measurable improvement. However, both models continued to misclassify the two spam examples as ham, indicating that spam recall remains limited.

Overall, LoRA fine-tuning improved performance on this task while maintaining parameter efficiency, but additional training data, class balancing, or longer training may further improve spam detection performance.
